# LLM classification finetuning
Hi! it's me Oskar or better known as 'iXpleew' and today we are doing the LLM classification ine-tuning problem! These libraries below I will be using in that notebook

In [ ]:
import numpy as np
import os
import pandas as pd
import xgboost as XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

First of all let's see sample of the dataset we are working with.

In [ ]:
TRAIN_SET = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/train.csv")
TRAIN_SET = TRAIN_SET.set_index("id")
TRAIN_SET2 = pd.DataFrame()

TEST_SET = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/test.csv")
TEST_SET2 = pd.DataFrame()
print(TEST_SET.head())

Let's add some traits to make model easier to teach, but for now we need to add the column for final classification. I will add `who_won` column in which:
 - 0 is a `winner_tie` equals 1
 - 1 is a `winner_model_a` equals 1
 - 2 is a `winner_model_b` equals 1
`who_won` will be a y list for classification 

In [ ]:
who_won = []
for amodel_bit, bmodel_bit, tie in zip(TRAIN_SET["winner_model_a"], TRAIN_SET["winner_model_b"],TRAIN_SET["winner_tie"]):
    if tie == 1:
        who_won.append(0)
    elif amodel_bit == 1:
        who_won.append(1)
    else:
        who_won.append(2)

## EDA
In this chapter we will do a bit of a EDA. First of all we will see if `model_a` and `model_b` are sets with the same models in them. It will make easier creating features, because user can prefer answers from 'better' models (like Claude or ChatGPT)

In [ ]:
models_a = set(TRAIN_SET["model_a"])
models_b = set(TRAIN_SET["model_b"])

print(f"A models No. is {len(models_a)}")
print(f"B models No. is {len(models_b)}")

if models_a == models_b:
    print("These models ARE the same")
else:
    print("These models ARE NOT the same")

As we can see these models are the same which will make EDA more easier to implement. Plan for this competition is that I will create 2 model one of which is a transformer model for text and the second one is for classification for featured traits.

In [ ]:
refuse_list = ["as an ai", "i cannot", "i can't", "i refuse", "i'm sorry", "i apologize"]
refuses_counts_a = []
refuses_counts_b = []

for phrase in refuse_list:
    TRAIN_SET2[f"{phrase.replace(" ", "_")}_count_a"] = TRAIN_SET["response_a"].str.lower().str.count(phrase)
    TRAIN_SET2[f"{phrase.replace(" ", "_")}_count_b"] = TRAIN_SET["response_b"].str.lower().str.count(phrase)

len_diff = TRAIN_SET["response_a"].str.len() - TRAIN_SET["response_b"].str.len()
a_longer = len_diff > 0
the_same = len_diff == 0
b_longer = len_diff < 0
response_values = [0,1,2]
longer_response = np.select([the_same, a_longer, b_longer], response_values)
TRAIN_SET2["longer_response"] = longer_response

print(TRAIN_SET2.info())

Now having a complete `TRAIN_SET2` I will split it into train set and validate to optimize the first model

## Model
let's create our first demo model o predict which AI will prefer user based on first feature engineered traits. I will also create a validate set here, to have a closer look with our output. The final score is measured in log loss

In [ ]:
for phrase in refuse_list:
    TEST_SET2[f"{phrase.replace(" ", "_")}_count_a"] = TEST_SET["response_a"].str.lower().str.count(phrase)
    TEST_SET2[f"{phrase.replace(" ", "_")}_count_b"] = TEST_SET["response_b"].str.lower().str.count(phrase)

len_diff = TEST_SET["response_a"].str.len() - TEST_SET["response_b"].str.len()
a_longer = len_diff > 0
the_same = len_diff == 0
b_longer = len_diff < 0
response_values = [0,1,2]
longer_response = np.select([the_same, a_longer, b_longer], response_values)
TEST_SET2["longer_response"] = longer_response


bst = XGBClassifier(n_estimators=350, learning_rate=0.3)
bst.fit(TRAIN_SET2, who_won)
predictions = bst.predict_proba(TEST_SET2)

submit = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv")
submit["winner_tie"] = predictions[:, 0]
submit["winner_model_a"] = predictions[:, 1]
submit["winner_model_b"] = predictions[:, 2]
submit.to_csv("submission.csv", index=False)